# Telugu QA - Domain LoRA Training (Combined 3-Domain)

Train LoRA adapters on **MuRIL** and **mBERT** using ALL 3 domains (government + literature + news) as a single combined dataset.

---

## 📦 Before You Start: Prerequisites

### 1. Base Model Checkpoints Required
You must have already trained base models using one of:
- `01_training_colab.ipynb` (MuRIL/IndicBERT)
- `03_train_mbert_colab.ipynb` (mBERT)

These checkpoints should be in your Google Drive at:
- MuRIL: `/content/drive/MyDrive/telugu-qa-checkpoints/muril/checkpoint-9000`
- mBERT: `/content/drive/MyDrive/telugu-qa-mBERT-checkpoints/mbert/checkpoint-13000`

### 2. Domain Data Zip
Run locally to create domain training zip:
```bash
python scripts/export_domain_for_colab.py
```

This creates `domain-colab.zip` containing combined domain data.

---

## 🎯 Adding Your Own Domain Data

### Step 1: Create SQuAD-format JSON
```json
{
  "version": "1.0",
  "data": [{
    "title": "Article Title",
    "paragraphs": [{
      "context": "Your Telugu paragraph here...",
      "qas": [{
        "id": "unique_id_001",
        "question": "మీ ప్రశ్న?",
        "answers": [{"text": "సమాధానం", "answer_start": 10}]
      }]
    }]
  }]
}
```

### Step 2: Place in data/domain/
```
data/domain/your_domain/
├── train.json       # Training QA pairs
└── validation.json  # Validation QA pairs
```

### Step 3: Verify and Export
```bash
python scripts/verify_squad.py data/domain/your_domain/train.json
python scripts/export_domain_for_colab.py --domain your_domain
```

---

## Plan
- **Run 1**: MuRIL + LoRA on combined domain data -> `muril-domain` adapter
- **Run 2**: mBERT + LoRA on combined domain data -> `mbert-domain` adapter

## Final App Models (4 total)
| Model | Type | Description |
|-------|------|------------|
| MuRIL | Base | Fine-tuned on TeQuAD (F1=79.2) |
| mBERT | Base | Fine-tuned on TeQuAD (F1=72.3) |
| **MuRIL-Domain** | **LoRA** | **+ 3-domain training (gov+lit+news)** |
| **mBERT-Domain** | **LoRA** | **+ 3-domain training (gov+lit+news)** |

## Data
| Domain | QA Pairs |
|--------|----------|
| Government | 9,623 |
| News | 7,761 |
| Literature | 2,701 |
| **Combined Train** | **19,055** |
| **Validation** | **1,030** |

---

## 1. Setup Environment

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers==4.44.0 datasets accelerate sentencepiece pyyaml
!pip install -q peft==0.10.0 bitsandbytes
print("\n✅ Done!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Output directory for trained adapters
DRIVE_OUTPUT = '/content/drive/MyDrive/telugu-qa-lora-adapters'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# TeQuAD-fine-tuned checkpoints already on Google Drive
CHECKPOINT_PATHS = {
    "muril": "/content/drive/MyDrive/telugu-qa-checkpoints/muril/checkpoint-9000",
    "mbert": "/content/drive/MyDrive/telugu-qa-mBERT-checkpoints/mbert/checkpoint-13000",
}

print(f"Adapter output: {DRIVE_OUTPUT}")
print()
for name, path in CHECKPOINT_PATHS.items():
    if os.path.exists(path):
        print(f"  {name}: Found at {path}")
    else:
        print(f"  {name}: NOT FOUND at {path}")

### Re-upload Checkpoints to Google Drive (if missing)

Run this cell **only if** the checkpoint paths above show "NOT FOUND".

You need to upload **2 files per model** from your local machine:
- `config.json` (~1 KB)
- `model.safetensors` (~900 MB for MuRIL, ~676 MB for mBERT)

These are in your local `models/checkpoints/muril/checkpoint-9000/` and `models/checkpoints/mbert/checkpoint-13000/` folders.

In [ ]:
#@title Upload Checkpoints to Google Drive { run: "auto" }
#@markdown Select which checkpoint to upload. Run once per model.
UPLOAD_MODEL = "muril" #@param ["muril", "mbert"]

from google.colab import files
import shutil

# Target paths on Drive
target_paths = {
    "muril": "/content/drive/MyDrive/telugu-qa-checkpoints/muril/checkpoint-9000",
    "mbert": "/content/drive/MyDrive/telugu-qa-mBERT-checkpoints/mbert/checkpoint-13000",
}

target = target_paths[UPLOAD_MODEL]
os.makedirs(target, exist_ok=True)

# Check what's already there
existing = os.listdir(target) if os.path.exists(target) else []
print(f"Target: {target}")
print(f"Existing files: {existing}")

needed = []
if "config.json" not in existing:
    needed.append("config.json")
if "model.safetensors" not in existing:
    needed.append("model.safetensors")

if not needed:
    print(f"\n✅ {UPLOAD_MODEL} checkpoint already complete! Skip this cell.")
else:
    print(f"\nMissing files: {needed}")
    print(f"Upload these files from your local models/checkpoints/{UPLOAD_MODEL}/ folder:\n")

    uploaded = files.upload()

    for filename, content in uploaded.items():
        dest = os.path.join(target, filename)
        # files.upload() saves to current dir, move to Drive
        if os.path.exists(f"/content/{filename}"):
            shutil.move(f"/content/{filename}", dest)
        elif os.path.exists(filename):
            shutil.move(filename, dest)
        size_mb = os.path.getsize(dest) / 1024 / 1024
        print(f"  ✅ {filename} -> {dest} ({size_mb:.1f} MB)")

    # Verify
    final_files = os.listdir(target)
    print(f"\nFiles now in {target}:")
    for f in final_files:
        size = os.path.getsize(os.path.join(target, f)) / 1024 / 1024
        print(f"  {f}: {size:.1f} MB")

    has_config = "config.json" in final_files
    has_model = "model.safetensors" in final_files
    if has_config and has_model:
        print(f"\n✅ {UPLOAD_MODEL} checkpoint is ready!")
    else:
        missing = []
        if not has_config: missing.append("config.json")
        if not has_model: missing.append("model.safetensors")
        print(f"\n⚠️  Still missing: {missing} — re-run this cell to upload them")

## 2. Upload Domain Data

Upload `domain-qa-colab-20260222_140839.zip` (contains combined train + validation + per-domain test sets)

**Checkpoints** are loaded directly from Google Drive:
- MuRIL (F1=79.2): `telugu-qa-checkpoints/muril/checkpoint-9000/`
- mBERT (F1=72.3): `telugu-qa-mBERT-checkpoints/mbert/checkpoint-13000/`

In [ ]:
from google.colab import files
import zipfile

print("Upload your domain-qa-colab-*.zip file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content')
        print(f"\n Extracted {filename}")

DATA_DIR = '/content/domain_data'
!ls -la {DATA_DIR}

In [ ]:
import json

# Verify combined dataset
for fname, label in [('domain_all_train.json', 'TRAIN'), ('domain_all_val.json', 'VALIDATION')]:
    path = f"{DATA_DIR}/{fname}"
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        paragraphs = data['data'][0]['paragraphs']
        qa_count = sum(len(p.get('qas', [])) for p in paragraphs)
        valid = 0
        for p in paragraphs:
            for qa in p.get('qas', []):
                if qa.get('answers'):
                    a = qa['answers'][0]
                    if p['context'][a['answer_start']:a['answer_start']+len(a['text'])] == a['text']:
                        valid += 1
        print(f"{label}: {qa_count} QA pairs, {len(paragraphs)} contexts, {valid}/{qa_count} valid spans")
    else:
        print(f"{label}: FILE NOT FOUND")

## 3. Select Model

Change this dropdown between Run 1 and Run 2.

**These are your TeQuAD-fine-tuned checkpoints**, not raw HuggingFace models.

In [ ]:
#@title Select Base Model { run: "auto" }
#@markdown **Run 1**: muril (TeQuAD F1=79.2) | **Run 2**: mbert (TeQuAD F1=72.3)
MODEL_NAME = "muril" #@param ["muril", "mbert"]

# Load from TeQuAD-fine-tuned checkpoint (NOT raw HuggingFace pretrained)
BASE_MODEL = CHECKPOINT_PATHS[MODEL_NAME]

print(f"\n Training: {MODEL_NAME} + LoRA on combined 3-domain data")
print(f"   Checkpoint: {BASE_MODEL}")

assert os.path.exists(BASE_MODEL), f"Checkpoint not found: {BASE_MODEL}. Upload to Google Drive first!"

In [ ]:
# Training + LoRA config
CONFIG = {
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "target_modules": ["query", "value"],
    # Training (T4 GPU optimized)
    "learning_rate": 2e-4,
    "num_train_epochs": 5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    # QA
    "max_seq_length": 384,
    "doc_stride": 128,
    # Checkpoints
    "save_steps": 500,
    "eval_steps": 500,
    "logging_steps": 50,
}

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4. Load TeQuAD-Fine-Tuned Model + Apply LoRA

Loads the **already-trained QA model** (not the raw pretrained one) so LoRA domain training builds on top of existing QA ability.

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, AutoConfig
import torch

# Original HuggingFace model names (for tokenizer + config fallback)
HF_MODEL = {
    "muril": "google/muril-base-cased",
    "mbert": "google/bert-base-multilingual-cased",
}[MODEL_NAME]

# Debug: list checkpoint files
print(f"Checkpoint dir: {BASE_MODEL}")
ckpt_files = os.listdir(BASE_MODEL)
print(f"Files found: {ckpt_files}")

# If config.json is missing from checkpoint, copy it from HuggingFace
if "config.json" not in ckpt_files:
    print(f"\n⚠️  config.json not found in checkpoint — downloading from {HF_MODEL}")
    config = AutoConfig.from_pretrained(HF_MODEL)
    config.save_pretrained(BASE_MODEL)
    print(f"   Saved config.json to {BASE_MODEL}")
else:
    print(f"   config.json found in checkpoint")

# Tokenizer from HuggingFace (identical — fine-tuning doesn't modify tokenizer)
print(f"\nLoading tokenizer from: {HF_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)

# Model weights from YOUR TeQuAD-fine-tuned checkpoint
# IMPORTANT: Load in float32! The Trainer's fp16 flag handles mixed-precision casting.
# Loading in float16 directly causes "Attempting to unscale FP16 gradients" error.
print(f"Loading model from checkpoint: {BASE_MODEL}")
model = AutoModelForQuestionAnswering.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
)
print(f"\n✅ Model loaded from checkpoint (float32)! Parameters: {model.num_parameters():,}")
print(f"   This model already has QA ability (TeQuAD-trained). LoRA will add domain knowledge.")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.QUESTION_ANS,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
)
model = get_peft_model(model, lora_config)

# Ensure ALL trainable params are float32 (prevents fp16 gradient unscale error)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("✅ LoRA Applied!")
model.print_trainable_parameters()
print(f"   Trainable param dtype: {next(p for p in model.parameters() if p.requires_grad).dtype}")

## 5. Pre-Training Baseline (Before Domain Training)

Evaluate the base model on **full per-domain test sets** (230 QA pairs total) and the **combined validation set** (1,030 QA pairs) using proper **F1** and **Exact Match** metrics.

This captures the model's performance BEFORE domain LoRA training, so we can measure improvement after.

In [ ]:
import re
import collections

# ============================================================
# F1 / Exact Match metric functions (adapted for Telugu)
# ============================================================

def normalize_telugu(text):
    """Normalize Telugu text for evaluation: strip whitespace, punctuation."""
    text = text.strip()
    # Remove punctuation (keep Telugu characters, digits, spaces)
    text = re.sub(r'[।,.;:!?"\'\-(){}[\]/<>@#$%^&*+=~`|\\]', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_tokens(text):
    """Tokenize by whitespace after normalization."""
    normalized = normalize_telugu(text)
    if not normalized:
        return []
    return normalized.split()


def compute_exact_match(prediction, ground_truth):
    """Exact match after normalization."""
    return int(normalize_telugu(prediction) == normalize_telugu(ground_truth))


def compute_f1(prediction, ground_truth):
    """Token-level F1 score."""
    pred_tokens = get_tokens(prediction)
    gold_tokens = get_tokens(ground_truth)

    if not gold_tokens and not pred_tokens:
        return 1.0
    if not gold_tokens or not pred_tokens:
        return 0.0

    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1


# ============================================================
# QA inference helper
# ============================================================

def run_qa(question, context, model, tokenizer):
    """Run QA inference and return predicted answer."""
    inputs = tokenizer(question, context, return_tensors="pt", max_length=384, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    if end_idx < start_idx:
        end_idx = start_idx
    answer = tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx+1], skip_special_tokens=True)
    return answer.strip()


# ============================================================
# Full dataset evaluation
# ============================================================

def load_test_examples(filepath):
    """Load SQuAD-format JSON into list of (question, context, answer) tuples."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    examples = []
    for article in data.get('data', []):
        for paragraph in article.get('paragraphs', []):
            context = paragraph['context']
            for qa in paragraph.get('qas', []):
                if not qa.get('answers'):
                    continue
                a = qa['answers'][0]
                # Validate span
                if context[a['answer_start']:a['answer_start']+len(a['text'])] == a['text']:
                    examples.append({
                        'question': qa['question'],
                        'context': context,
                        'answer': a['text'],
                    })
    return examples


def evaluate_dataset(examples, model, tokenizer, label="Dataset"):
    """Evaluate model on a list of QA examples. Returns F1, EM, and per-example predictions."""
    total_f1 = 0.0
    total_em = 0
    predictions = []

    for ex in examples:
        pred = run_qa(ex['question'], ex['context'], model, tokenizer)
        f1 = compute_f1(pred, ex['answer'])
        em = compute_exact_match(pred, ex['answer'])
        total_f1 += f1
        total_em += em
        predictions.append({
            'question': ex['question'],
            'expected': ex['answer'],
            'predicted': pred,
            'f1': f1,
            'em': em,
        })

    n = len(examples)
    avg_f1 = (total_f1 / n * 100) if n > 0 else 0
    avg_em = (total_em / n * 100) if n > 0 else 0
    return {'f1': avg_f1, 'em': avg_em, 'count': n, 'predictions': predictions}


# ============================================================
# Load all test/val sets
# ============================================================

DOMAIN_TEST_FILES = {
    "Government": f"{DATA_DIR}/government_test.json",
    "Literature": f"{DATA_DIR}/literature_test.json",
    "News": f"{DATA_DIR}/news_test.json",
}
VAL_FILE = f"{DATA_DIR}/domain_all_val.json"

domain_test_sets = {}
for domain, path in DOMAIN_TEST_FILES.items():
    if os.path.exists(path):
        examples = load_test_examples(path)
        domain_test_sets[domain] = examples
        print(f"  Loaded {domain} test: {len(examples)} QA pairs")
    else:
        print(f"  {domain} test file not found: {path}")

val_examples = load_test_examples(VAL_FILE) if os.path.exists(VAL_FILE) else []
print(f"  Loaded validation set: {len(val_examples)} QA pairs")


# ============================================================
# BEFORE training: evaluate base model
# ============================================================

print(f"\n{'='*70}")
print(f"  BASELINE EVALUATION (BEFORE domain training) - {MODEL_NAME}")
print(f"{'='*70}")

baseline_scores = {}

# Per-domain test sets
for domain, examples in domain_test_sets.items():
    result = evaluate_dataset(examples, model, tokenizer, domain)
    baseline_scores[domain] = result
    print(f"  {domain:12s}  F1={result['f1']:5.1f}  EM={result['em']:5.1f}  ({result['count']} QA pairs)")

# Combined validation set
if val_examples:
    val_result = evaluate_dataset(val_examples, model, tokenizer, "Validation")
    baseline_scores['Combined_Val'] = val_result
    print(f"  {'Combined Val':12s}  F1={val_result['f1']:5.1f}  EM={val_result['em']:5.1f}  ({val_result['count']} QA pairs)")

print(f"\n  Baseline scores saved. Will compare after training.")
print(f"{'='*70}")


# ============================================================
# Qualitative examples (6 handpicked domain questions)
# ============================================================

DOMAIN_TEST_CASES = [
    {"domain": "Government",
     "context": "వైద్య ఆరోగ్య శాఖ రైతు బంధు పథకాన్ని హైదరాబాద్ జిల్లాలో ప్రారంభించింది. ఈ పథకం ద్వారా అర్హులైన లబ్ధిదారులకు రూ.5,000 ఆర్థిక సహాయం అందజేయబడుతుంది.",
     "question": "రైతు బంధు పథకం ద్వారా ఎంత సహాయం అందుతుంది?",
     "expected": "రూ.5,000"},
    {"domain": "Government",
     "context": "ఆదాయ ధృవీకరణ పత్రం పొందడం కోసం హైదరాబాద్ జిల్లా మీసేవ కేంద్రంలో దరఖాస్తు చేసుకోవచ్చు. ఈ సర్టిఫికేట్ 15 రోజులు లో జారీ చేయబడుతుంది.",
     "question": "ఆదాయ ధృవీకరణ పత్రం ఎన్ని రోజుల్లో వస్తుంది?",
     "expected": "15 రోజులు"},
    {"domain": "News",
     "context": "హైదరాబాద్లో భారతీయ జనతా పార్టీ ముఖ్యమంత్రి బహిరంగ సభ నిర్వహించారు. ఈ సందర్భంగా రాష్ట్ర అభివృద్ధిపై ప్రధానంగా మాట్లాడారు.",
     "question": "బహిరంగ సభ ఎక్కడ జరిగింది?",
     "expected": "హైదరాబాద్"},
    {"domain": "News",
     "context": "క్రికెట్: విరాట్ కోహ్లి అద్భుతమైన ప్రదర్శనతో సెంచరీ సాధించారు. ఈ బ్యాట్స్‌మన్ విశాఖపట్నంలో జరిగిన టెస్ట్ మ్యాచ్లో 156 రన్స్ తేడాతో గెలిచారు.",
     "question": "ఈ మ్యాచ్ ఎక్కడ జరిగింది?",
     "expected": "విశాఖపట్నం"},
    {"domain": "Literature",
     "context": "ఉప్పు కప్పురమ్ము ఒక్క పోలిక నుండు చూడ చూడ రుచుల జాడ లేరు పురుషులందు పుణ్య పురుషు లెఱుంగరు విశ్వదాభిరామ వినురవేమ. ఈ పద్యం వేమన రచించారు.",
     "question": "ఈ పద్యం రచయిత ఎవరు?",
     "expected": "వేమన"},
    {"domain": "Literature",
     "context": "పోతన 15వ శతాబ్దంకు చెందిన ప్రసిద్ధ తెలుగు కవి. కావ్య కవి అనే బిరుదు కలిగిన ఈ కవి ఆంధ్ర మహాభాగవతము అనే గొప్ప రచన చేశారు.",
     "question": "పోతన ప్రసిద్ధ రచన ఏది?",
     "expected": "ఆంధ్ర మహాభాగవతము"},
]

print(f"\nQualitative Examples (6 handpicked questions) - BEFORE training:")
for tc in DOMAIN_TEST_CASES:
    pred = run_qa(tc['question'], tc['context'], model, tokenizer)
    f1 = compute_f1(pred, tc['expected'])
    status = 'PASS' if f1 > 0.5 else 'FAIL'
    print(f"  [{status}] {tc['domain']:10s} Q: {tc['question'][:50]}...")
    print(f"         Expected: {tc['expected']}")
    print(f"         Got:      {pred}  (F1={f1:.2f})")

## 6. Load & Preprocess Combined Data

In [ ]:
from datasets import Dataset

def load_squad_json(filepath):
    """Load SQuAD-format JSON with span validation."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    examples = []
    for article in data.get('data', []):
        for paragraph in article.get('paragraphs', []):
            context = paragraph['context']
            for qa in paragraph.get('qas', []):
                if not qa.get('answers'):
                    continue
                a = qa['answers'][0]
                if context[a['answer_start']:a['answer_start']+len(a['text'])] == a['text']:
                    examples.append({
                        'id': qa['id'],
                        'context': context,
                        'question': qa['question'],
                        'answers': {
                            'text': [a['text']],
                            'answer_start': [a['answer_start']]
                        }
                    })
    return Dataset.from_list(examples)


# Load COMBINED dataset (all 3 domains merged)
print("Loading combined domain training data...")
train_dataset = load_squad_json(f"{DATA_DIR}/domain_all_train.json")
val_dataset = load_squad_json(f"{DATA_DIR}/domain_all_val.json")

print(f"Train: {len(train_dataset)} examples")
print(f"Val:   {len(val_dataset)} examples")

In [ ]:
def preprocess_function(examples):
    """Preprocess QA examples for transformer training."""
    questions = [q.strip() for q in examples["question"]]
    contexts = examples["context"]
    
    tokenized = tokenizer(
        questions, contexts,
        max_length=CONFIG["max_seq_length"],
        truncation="only_second",
        stride=CONFIG["doc_stride"],
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    start_positions = []
    end_positions = []
    
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        answers = examples["answers"][sample_idx]
        
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
        
        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])
        sequence_ids = tokenized.sequence_ids(i)
        
        # Find context token range
        context_start = 0
        while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
            context_start += 1
        context_end = len(sequence_ids) - 1
        while context_end >= 0 and sequence_ids[context_end] != 1:
            context_end -= 1
        
        if context_start >= len(sequence_ids) or context_end < 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
        
        # Check if answer is within this chunk
        if offsets[context_start][0] > answer_end or offsets[context_end][1] < answer_start:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= answer_start:
                idx += 1
            start_positions.append(idx - 1)
            
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= answer_end:
                idx -= 1
            end_positions.append(idx + 1)
    
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


print("Preprocessing...")
train_tokenized = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

print(f"Train tokens: {len(train_tokenized)} samples")
print(f"Val tokens:   {len(val_tokenized)} samples")

## 7. Train

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

# Fix: ensure model is float32 before training (prevents fp16 gradient scaler crash)
model = model.float()
for param in model.parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
print(f"Model dtype check: {next(model.parameters()).dtype}")

output_dir = f"{DRIVE_OUTPUT}/{MODEL_NAME}-domain"

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir=f"{output_dir}/logs",
    logging_steps=CONFIG["logging_steps"],
    report_to="none",
    fp16=False,
    dataloader_num_workers=2,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

print(f"\nTraining {MODEL_NAME} + LoRA on ~19K domain QA pairs...")
print(f"Output: {output_dir}")
print(f"\n>>> Run the NEXT cell to start training <<<")

In [ ]:
train_result = trainer.train()

print("\n" + "="*50)
print(f"Training Results ({MODEL_NAME}-domain):")
print("="*50)
metrics = train_result.metrics
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

In [ ]:
eval_results = trainer.evaluate()
print(f"\nEval Results ({MODEL_NAME}-domain):")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## 8. Post-Training Evaluation + Comparative Results

Run the **same F1/EM evaluation** on per-domain test sets and the combined validation set AFTER training.

Produces a side-by-side comparison table proving domain training improvement.

In [ ]:
# ============================================================
# AFTER training: evaluate domain-trained model (same tests)
# ============================================================

print(f"\n{'='*70}")
print(f"  POST-TRAINING EVALUATION (AFTER domain training) - {MODEL_NAME}")
print(f"{'='*70}")

after_scores = {}

# Per-domain test sets
for domain, examples in domain_test_sets.items():
    result = evaluate_dataset(examples, model, tokenizer, domain)
    after_scores[domain] = result
    print(f"  {domain:12s}  F1={result['f1']:5.1f}  EM={result['em']:5.1f}  ({result['count']} QA pairs)")

# Combined validation set
if val_examples:
    val_result = evaluate_dataset(val_examples, model, tokenizer, "Validation")
    after_scores['Combined_Val'] = val_result
    print(f"  {'Combined Val':12s}  F1={val_result['f1']:5.1f}  EM={val_result['em']:5.1f}  ({val_result['count']} QA pairs)")


# ============================================================
# COMPARISON TABLE: Before vs After
# ============================================================

print(f"\n\n{'='*70}")
print(f"  COMPARATIVE RESULTS: {MODEL_NAME} Base vs {MODEL_NAME}-Domain (LoRA)")
print(f"{'='*70}")
print(f"")
print(f"  {'Dataset':<15} {'Metric':<8} {'BEFORE':>8} {'AFTER':>8} {'Change':>10}")
print(f"  {'-'*15} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")

all_domains_list = list(domain_test_sets.keys()) + (['Combined_Val'] if 'Combined_Val' in baseline_scores else [])

comparison_data = {}
for ds_name in all_domains_list:
    display_name = ds_name if ds_name != 'Combined_Val' else 'All Domains'
    b = baseline_scores.get(ds_name, {})
    a = after_scores.get(ds_name, {})
    if not b or not a:
        continue

    for metric in ['f1', 'em']:
        before_val = b[metric]
        after_val = a[metric]
        change = after_val - before_val
        sign = '+' if change >= 0 else ''
        metric_label = 'F1' if metric == 'f1' else 'EM'
        print(f"  {display_name:<15} {metric_label:<8} {before_val:>7.1f}% {after_val:>7.1f}% {sign}{change:>8.1f}%")

    comparison_data[ds_name] = {
        'before_f1': b['f1'], 'after_f1': a['f1'],
        'before_em': b['em'], 'after_em': a['em'],
        'f1_change': a['f1'] - b['f1'],
        'em_change': a['em'] - b['em'],
        'count': b['count'],
    }
    print(f"  {'':<15} {'':<8} {'':<8} {'':<8}")

print(f"  {'-'*15} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")

# Overall average across per-domain test sets
domain_only = [k for k in comparison_data if k != 'Combined_Val']
if domain_only:
    avg_before_f1 = sum(comparison_data[k]['before_f1'] for k in domain_only) / len(domain_only)
    avg_after_f1 = sum(comparison_data[k]['after_f1'] for k in domain_only) / len(domain_only)
    avg_before_em = sum(comparison_data[k]['before_em'] for k in domain_only) / len(domain_only)
    avg_after_em = sum(comparison_data[k]['after_em'] for k in domain_only) / len(domain_only)
    f1_diff = avg_after_f1 - avg_before_f1
    em_diff = avg_after_em - avg_before_em
    sign_f1 = '+' if f1_diff >= 0 else ''
    sign_em = '+' if em_diff >= 0 else ''
    print(f"  {'AVERAGE':<15} {'F1':<8} {avg_before_f1:>7.1f}% {avg_after_f1:>7.1f}% {sign_f1}{f1_diff:>8.1f}%")
    print(f"  {'(test sets)':<15} {'EM':<8} {avg_before_em:>7.1f}% {avg_after_em:>7.1f}% {sign_em}{em_diff:>8.1f}%")

print(f"\n{'='*70}")


# ============================================================
# Qualitative examples - AFTER training
# ============================================================

print(f"\nQualitative Examples (6 handpicked questions) - AFTER training:")
qual_before_correct = 0
qual_after_correct = 0
for tc in DOMAIN_TEST_CASES:
    pred = run_qa(tc['question'], tc['context'], model, tokenizer)
    f1 = compute_f1(pred, tc['expected'])
    status = 'PASS' if f1 > 0.5 else 'FAIL'
    if f1 > 0.5:
        qual_after_correct += 1
    print(f"  [{status}] {tc['domain']:10s} Q: {tc['question'][:50]}...")
    print(f"         Expected: {tc['expected']}")
    print(f"         Got:      {pred}  (F1={f1:.2f})")

print(f"\n  Qualitative score: {qual_after_correct}/6")


# ============================================================
# Show 5 best improvements + 5 worst from per-domain test sets
# ============================================================

print(f"\n{'='*70}")
print(f"  TOP 5 IMPROVEMENTS (biggest F1 gains)")
print(f"{'='*70}")

improvements = []
for domain in domain_test_sets:
    b_preds = baseline_scores.get(domain, {}).get('predictions', [])
    a_preds = after_scores.get(domain, {}).get('predictions', [])
    for bp, ap in zip(b_preds, a_preds):
        delta = ap['f1'] - bp['f1']
        improvements.append({
            'domain': domain,
            'question': bp['question'],
            'expected': bp['expected'],
            'before_pred': bp['predicted'],
            'after_pred': ap['predicted'],
            'before_f1': bp['f1'],
            'after_f1': ap['f1'],
            'delta': delta,
        })

improvements.sort(key=lambda x: x['delta'], reverse=True)
for imp in improvements[:5]:
    print(f"  [{imp['domain']}] F1: {imp['before_f1']:.2f} -> {imp['after_f1']:.2f} (+{imp['delta']:.2f})")
    print(f"    Q: {imp['question'][:60]}...")
    print(f"    Expected: {imp['expected']}")
    print(f"    Before:   {imp['before_pred']}")
    print(f"    After:    {imp['after_pred']}")
    print()

if any(imp['delta'] < 0 for imp in improvements):
    print(f"  REGRESSIONS (if any):")
    regressions = [imp for imp in improvements if imp['delta'] < 0]
    regressions.sort(key=lambda x: x['delta'])
    for imp in regressions[:3]:
        print(f"  [{imp['domain']}] F1: {imp['before_f1']:.2f} -> {imp['after_f1']:.2f} ({imp['delta']:.2f})")
        print(f"    Q: {imp['question'][:60]}...")

## 9. Save LoRA Adapter

In [ ]:
adapter_path = f"{output_dir}/adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Save training config + comprehensive results
config_to_save = {
    "base_model": BASE_MODEL,
    "model_name": MODEL_NAME,
    "domains": ["government", "literature", "news"],
    "training_type": "combined_3_domain",
    **CONFIG,
    "training_samples": len(train_tokenized),
    "validation_samples": len(val_tokenized),
    "final_train_loss": metrics.get("train_loss"),
    "final_eval_loss": eval_results.get("eval_loss"),
    # Per-domain F1/EM comparison
    "evaluation": {
        ds: {
            "before": {"f1": comparison_data[ds]["before_f1"], "em": comparison_data[ds]["before_em"]},
            "after": {"f1": comparison_data[ds]["after_f1"], "em": comparison_data[ds]["after_em"]},
            "improvement": {"f1": comparison_data[ds]["f1_change"], "em": comparison_data[ds]["em_change"]},
            "test_count": comparison_data[ds]["count"],
        }
        for ds in comparison_data
    },
}

with open(f"{adapter_path}/training_config.json", 'w') as f:
    json.dump(config_to_save, f, ensure_ascii=False, indent=2)

# Also save the full comparison as a separate file for easy reference
with open(f"{adapter_path}/evaluation_comparison.json", 'w') as f:
    json.dump({
        "model": f"{MODEL_NAME}-domain",
        "base_model": BASE_MODEL,
        "comparison": comparison_data,
    }, f, ensure_ascii=False, indent=2)

adapter_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"\n Saved: {adapter_path}")
print(f"   Size: {adapter_size / 1024 / 1024:.2f} MB")
print(f"   Model: {MODEL_NAME}-domain")
print(f"   Domains: government + literature + news")
print(f"   Includes: training_config.json + evaluation_comparison.json")

## 10. Download Adapter

In [ ]:
import shutil

zip_name = f"{MODEL_NAME}_domain_lora_adapter"
shutil.make_archive(f"/content/{zip_name}", 'zip', adapter_path)

print(f"Download: /content/{zip_name}.zip")
from google.colab import files
files.download(f"/content/{zip_name}.zip")

---

## Training Sequence

Run this notebook **2 times**:

| Run | MODEL_NAME | Checkpoint (on Google Drive) | Output Adapter |
|-----|-----------|------------------------------|----------------|
| 1 | `muril` | `telugu-qa-checkpoints/muril/checkpoint-9000` | `muril-domain/adapter/` |
| 2 | `mbert` | `telugu-qa-mBERT-checkpoints/mbert/checkpoint-13000` | `mbert-domain/adapter/` |

**Between runs:**
1. Change `MODEL_NAME` in Section 3
2. Runtime -> **Restart runtime**
3. Re-run all cells from Section 1

---

## After Both Runs

Your Google Drive will have:
```
telugu-qa-lora-adapters/
  muril-domain/adapter/    <- MuRIL LoRA trained on all 3 domains
  mbert-domain/adapter/    <- mBERT LoRA trained on all 3 domains
```

Download both adapters and place locally:
```
models/adapters/
  muril-domain/   <- muril-domain adapter files
  mbert-domain/   <- mbert-domain adapter files
```

## Final 4 Models in App
```
1. MuRIL (base)       <- models/checkpoints/muril/
2. mBERT (base)       <- models/checkpoints/mbert/
3. MuRIL-Domain (LoRA)<- models/adapters/muril-domain/
4. mBERT-Domain (LoRA)<- models/adapters/mbert-domain/
```

Compare base vs domain-trained models on domain questions!